# Tugas Software Testing — Kuliah Asinkron

| | |
|---|---|
| **Nama** | Anugrah Putra Al Fatih |
| **NIM** | 26/59120/PPA/07385 |

**Cara mengerjakan.** Instruksi lengkap ada di dokumen *Kuliah Asinkron: Software Testing*. Ringkasnya:

* **Bagian A** – kerjakan Latihan **2, 3, 5, 6, dan 7** di bawah ini (Latihan 1, 4, dan 8 dianjurkan, tidak dinilai).
* **Bagian B** – mini-proyek `nilai_huruf` di akhir notebook.
* **Bagian C** – refleksi singkat di sel paling akhir.

Simpan notebook ini dengan nama `Tugas_Testing_<NIM>.ipynb`, jalankan **Restart & Run All**, lalu kumpulkan **beserta output-nya**. Bagian latihan memakai Bahasa Inggris agar sama dengan slide; jawaban boleh dalam Bahasa Indonesia.

# Software Testing — Hands-on Exercises (Python)

Companion notebook for the lecture **Software Testing** (adapted from I. Sommerville, *Engineering Software Products*, Ch. 9).

You will practise, in plain Python with the built-in `unittest` framework:

| # | Topic | Lecture slides |
|---|-------|----------------|
| 1 | Equivalence partitions | Unit testing, Fig. 9.3, Table 9.3 |
| 2 | Running and *reading* executable tests | Program 9.3 |
| 3 | Edge cases | Table 9.4 guidelines |
| 4 | Arrange – Action – Assert | Program 9.2 |
| 5 | Test-driven development | Fig. 9.8, Table 9.9 |
| 6 | Risk-based security testing | Table 9.11, risk analysis |
| 7 | Code review with a checklist | Table 9.13 |
| 8 | Running tests from files | Program 9.4 |

**How to use it.** Run the cells from top to bottom (Jupyter, VS Code or Google Colab — no extra packages needed). Cells marked **✏️ TODO** are for you. Every TODO cell runs without crashing even before you edit it; the test runner will simply report failures until your solution is right.

## 0. Setup

In [1]:
import re
import unittest

def run_tests(*test_cases, verbosity=2):
    """Run one or more unittest.TestCase classes inside the notebook and print a summary."""
    suite = unittest.TestSuite()
    for tc in test_cases:
        suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(tc))
    result = unittest.TextTestRunner(verbosity=verbosity).run(suite)
    print(f"\n==> {result.testsRun} run, {len(result.failures)} failures, {len(result.errors)} errors")
    return result

The function under test is **Program 9.1** from the lecture, copied unchanged (only spacing is tidied):

In [2]:
def namecheck(s):
    # Checks that a name only includes alphabetic characters, - or
    # a single quote. Names must be between 2 and 40 characters long
    # quoted strings and -- are disallowed
    namex = r"^[a-zA-Z][a-zA-Z-']{1,39}$"
    if re.match(namex, s):
        if re.search("'.*'", s) or re.search("--", s):
            return False
        else:
            return True
    else:
        return False

namecheck("O'Reilly"), namecheck("C-3PO")

(True, False)

## Exercise 1 — Equivalence partitions

Table 9.3 splits the inputs of `namecheck` into **2 correct** and **4 incorrect** partitions:

| Partition | Rule |
|---|---|
| `C1` Correct names 1 | only letters, 2–40 characters |
| `C2` Correct names 2 | letters, hyphens or apostrophes, 2–40 characters |
| `I1` Incorrect names 1 | 2–40 characters but disallowed characters |
| `I2` Incorrect names 2 | allowed characters but length 1 or > 40 |
| `I3` Incorrect names 3 | first character is a hyphen or an apostrophe |
| `I4` Incorrect names 4 | contains a double hyphen, quoted text, or both |

**✏️ TODO 1a.** For each input below, write the partition you think it belongs to *and* whether `namecheck` should return `True` or `False`. Then run the cell to compare your prediction with the real result.

In [3]:
# input            : (your partition, your predicted result)
predictions = {
    "Kurniawan"     : ("C1", True),
    "Mary-Jane"     : ("C2", True),
    "R2D2"          : ("I1", False),
    "'Connor"       : ("I3", False),
    "X"             : ("I2", False),
    "Smith--Jones"  : ("I4", False),
}

for name, (partition, predicted) in predictions.items():
    actual = namecheck(name)
    mark = "✓" if predicted == actual else "✗"
    print(f"{mark} {name!r:16} partition={partition:3} predicted={predicted!s:5} actual={actual}")

✓ 'Kurniawan'      partition=C1  predicted=True  actual=True
✓ 'Mary-Jane'      partition=C2  predicted=True  actual=True
✓ 'R2D2'           partition=I1  predicted=False actual=False
✓ "'Connor"        partition=I3  predicted=False actual=False
✓ 'X'              partition=I2  predicted=False actual=False
✓ 'Smith--Jones'   partition=I4  predicted=False actual=False


**✏️ TODO 1b.** Find **one more input per partition** that you have not seen yet in the lecture and add it to `my_examples`. The cell checks that the result matches the partition type (correct → `True`, incorrect → `False`).

In [4]:
my_examples = {
    "C1": "Budi",          # letters only
    "C2": "Anne-Marie",    # letters + hyphen
    "I1": "Ali@Baba",      # disallowed character '@'
    "I2": "A" * 41,        # allowed characters but 41 long
    "I3": "-Andi",         # starts with a hyphen
    "I4": "Jo'an'na",      # quoted text between two apostrophes
}

for part, name in my_examples.items():
    expected = part.startswith("C")
    status = "OK " if name and namecheck(name) == expected else "TODO"
    print(f"{status} {part}: {name!r}")

OK  C1: 'Budi'
OK  C2: 'Anne-Marie'
OK  I1: 'Ali@Baba'
OK  I2: 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA'
OK  I3: '-Andi'
OK  I4: "Jo'an'na"


## Exercise 2 — Run (and read!) the executable tests of Program 9.3

Below are the 14 tests of Program 9.3, unchanged. Run them.

In [5]:
class TestNameCheck(unittest.TestCase):

    def test_alphaname(self):
        self.assertTrue(namecheck('Sommerville'))

    def test_doublequote(self):
        self.assertFalse(namecheck("Thisis'maliciouscode'"))

    def test_namestartswithhyphen(self):
        self.assertFalse(namecheck('-Sommerville'))

    def test_namestartswithquote(self):
        self.assertFalse(namecheck("'Reilly"))

    def test_nametoolong(self):
        self.assertFalse(namecheck('Thisisalongstringwithmorethen40charactersfrombeginningtoend'))

    def test_nametooshort(self):
        self.assertFalse(namecheck('S'))

    def test_namewithdigit(self):
        self.assertFalse(namecheck('C-3PO'))

    def test_namewithdoublehyphen(self):
        self.assertFalse(namecheck('--badcode'))

    def test_namewithhyphen(self):
        self.assertTrue(namecheck('Washington-Wilson'))

    def test_namewithinvalidchar(self):
        self.assertFalse(namecheck('Sommer_ville'))

    def test_namewithquote(self):
        self.assertTrue(namecheck("O'Reilly"))

    def test_namewithspaces(self):
        self.assertFalse(namecheck('Washington Wilson'))

    def test_shortname(self):
        self.assertTrue('Sx')

    def test_thiswillfail(self):
        self.assertTrue(namecheck("O Reilly"))

run_tests(TestNameCheck);

test_alphaname (__main__.TestNameCheck) ... 

ok


test_doublequote (__main__.TestNameCheck) ... 

ok


test_namestartswithhyphen (__main__.TestNameCheck) ... 

ok


... (21 outputs left out)

FAIL


FAIL: test_thiswillfail (__main__.TestNameCheck)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_16180\65999436.py", line 43, in test_thiswillfail
    self.assertTrue(namecheck("O Reilly"))
AssertionError: False is not true



----------------------------------------------------------------------
Ran 14 tests in 0.011s

FAILED (failures=1)



==> 14 run, 1 failures, 0 errors


**Questions**

1. Which test fails, and why is that *expected*?
2. One test **can never fail**, whatever `namecheck` does. Which one, and why? *(Hint: what is `bool('Sx')`?)*

**✏️ TODO 2.** Fix the test that can never fail in the class below, then prove your fix works: temporarily break `namecheck` (the `broken_namecheck` cell) and check that your fixed test now catches the break.

**Jawaban**

1. Yang gagal adalah `test_thiswillfail`. `"O Reilly"` mengandung spasi, padahal spasi tidak ada di regex `[a-zA-Z-']`, sehingga `namecheck` mengembalikan `False`, sedangkan tes mengharapkan `True`. Kegagalan ini memang **disengaja** (namanya pun *this will fail*): tes ini dibuat untuk menunjukkan seperti apa laporan tes yang gagal. Yang salah di sini tesnya, bukan programnya.
2. `test_shortname` **tidak mungkin gagal** karena isinya `self.assertTrue('Sx')`. Fungsi `namecheck` tidak pernah dipanggil. String yang tidak kosong selalu *truthy* (`bool('Sx') == True`), jadi apa pun isi `namecheck`, tes ini tetap lulus. Perbaikannya: `self.assertTrue(namecheck('Sx'))`.

In [6]:
class TestShortName(unittest.TestCase):
    def test_shortname(self):
        self.assertTrue(namecheck('Sx'))   # fixed: actually calls namecheck

run_tests(TestShortName);

test_shortname (__main__.TestShortName) ... 

ok


----------------------------------------------------------------------
Ran 1 test in 0.001s

OK



==> 1 run, 0 failures, 0 errors


In [7]:
# Mutation check: a version of namecheck that wrongly rejects 2-character names.
_original = namecheck

def broken_namecheck(s):
    return _original(s) and len(s) > 2

namecheck = broken_namecheck          # swap in the broken version
try:
    print("With a BROKEN namecheck, a good test_shortname should FAIL:")
    run_tests(TestShortName, verbosity=1)
finally:
    namecheck = _original             # always restore the real function

F

FAIL: test_shortname (__main__.TestShortName)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_16180\705790997.py", line 3, in test_shortname
    self.assertTrue(namecheck('Sx'))   # fixed: actually calls namecheck
AssertionError: False is not true



----------------------------------------------------------------------
Ran 1 test in 0.000s

FAILED (failures=1)


With a BROKEN namecheck, a good test_shortname should FAIL:

==> 1 run, 1 failures, 0 errors


**Hasil mutation check:** dengan `broken_namecheck` (menolak nama 2 karakter), `test_shortname` yang sudah diperbaiki **gagal**. Jadi tes ini benar-benar bisa mendeteksi bug. Versi lama (`assertTrue('Sx')`) akan tetap lulus walaupun fungsinya rusak.

## Exercise 3 — Edge cases (Table 9.4 guidelines)

`namecheck` accepts names of **2 to 40** characters. The guideline *“test edge cases”* says to pick inputs **at** the boundaries, and *“don't forget null and zero”* says to try the empty string.

**✏️ TODO 3.** Complete the five tests. Use `'A' * n` to build a name of length `n`.

In [8]:
class TestNameCheckEdges(unittest.TestCase):

    def test_length_1_rejected(self):
        self.assertFalse(namecheck('A' * 1))

    def test_length_2_accepted(self):
        self.assertTrue(namecheck('A' * 2))

    def test_length_40_accepted(self):
        self.assertTrue(namecheck('A' * 40))

    def test_length_41_rejected(self):
        self.assertFalse(namecheck('A' * 41))

    def test_empty_string_rejected(self):
        self.assertFalse(namecheck(''))

run_tests(TestNameCheckEdges);

test_empty_string_rejected (__main__.TestNameCheckEdges) ... 

ok


test_length_1_rejected (__main__.TestNameCheckEdges) ... 

ok


test_length_2_accepted (__main__.TestNameCheckEdges) ... 

ok


... (2 outputs left out)

test_length_41_rejected (__main__.TestNameCheckEdges) ... 

ok


----------------------------------------------------------------------
Ran 5 tests in 0.005s

OK



==> 5 run, 0 failures, 0 errors


⚠️ An empty test body (`pass`) *passes*. A passing test only means something if it contains an assertion — the same lesson as Exercise 2.

**Discussion.** Run the next cell. Is `namecheck` right to reject these names? Is this a *programming* error or an *understanding* error (slide “Program bugs”)?

In [9]:
for name in ["D'Arcy-O'Neil", "Zoë", "José"]:
    print(f"{name!r:16} -> {namecheck(name)}")

"D'Arcy-O'Neil"  -> False
'Zoë'            -> False
'José'           -> False


**Jawaban diskusi.** Ketiga nama di atas adalah nama asli yang sah, tetapi semuanya **ditolak**:

* `D'Arcy-O'Neil` punya dua apostrof, jadi regex `'.*'` menganggapnya *quoted text* (aturan I4).
* `Zoë` dan `José` memakai huruf beraksen (non-ASCII) yang tidak termasuk `[a-zA-Z]`.

Kode menjalankan spesifikasinya dengan tepat. Jadi ini bukan *programming error*, melainkan **understanding error**: pembuat program salah memahami seperti apa "nama yang valid" di dunia nyata. Spesifikasinya yang perlu diperbaiki (misalnya mengizinkan huruf Unicode dan membedakan apostrof di dalam nama dari kode berbahaya), baru kemudian kodenya.

## Exercise 4 — Arrange, Action, Assert

Program 9.2 tests an `interest_calculator(p, r, n)` whose code is **not shown** in the lecture. Here we use our **own** simple definition so that you can practise the AAA pattern:

> `simple_interest(p, r, n)` = interest on principal `p` at an annual rate of `r` % for `n` days, using a 365-day year, rounded to 2 decimals.

*(This is not Sommerville's calculator — with this formula the inputs `p=17000, r=3, n=365` give 510.0, not the 270.36 expected in Program 9.2, so the book's function must use a different formula.)*

In [10]:
def simple_interest(p, r, n):
    return round(p * (r / 100) * (n / 365), 2)

**✏️ TODO 4.** Write three tests, each with clearly marked `# Arrange`, `# Action`, `# Assert` sections:

1. zero principal → interest 0
2. zero days → interest 0
3. one full year: `p = 1000`, `r = 5`, `n = 365` → 50.0

Bonus: `assertEqual` on floats is fragile. Look up `assertAlmostEqual` and use it in a fourth test with `n = 30`.

In [11]:
class TestSimpleInterest(unittest.TestCase):

    def test_zero_principal(self):
        # Arrange
        p, r, n = 0, 5, 365
        # Action
        interest = simple_interest(p, r, n)
        # Assert
        self.assertEqual(interest, 0)

    def test_zero_days(self):
        # Arrange
        p, r, n = 1000, 5, 0
        # Action
        interest = simple_interest(p, r, n)
        # Assert
        self.assertEqual(interest, 0)

    def test_one_year(self):
        # Arrange
        p, r, n = 1000, 5, 365
        # Action
        interest = simple_interest(p, r, n)
        # Assert
        self.assertEqual(interest, 50.0)

    def test_thirty_days_almost_equal(self):
        # Arrange
        p, r, n = 1000, 5, 30          # 1000 * 0.05 * 30/365 = 4.1095...
        # Action
        interest = simple_interest(p, r, n)
        # Assert
        self.assertAlmostEqual(interest, 4.11, places=2)

run_tests(TestSimpleInterest);

test_one_year (__main__.TestSimpleInterest) ... 

ok


test_thirty_days_almost_equal (__main__.TestSimpleInterest) ... 

ok


test_zero_days (__main__.TestSimpleInterest) ... 

ok


test_zero_principal (__main__.TestSimpleInterest) ... 

ok


----------------------------------------------------------------------
Ran 4 tests in 0.004s

OK



==> 4 run, 0 failures, 0 errors


## Exercise 5 — Test-driven development

Follow the TDD cycle from Table 9.9: **write a test → write a stub that fails → run all tests → implement → rerun → refactor.**

The goal is a function `average(values)` that:

* returns the arithmetic mean of a non-empty list of numbers;
* for a single value returns that value (*“one is different”*);
* raises `ValueError` for an empty list (*“don't forget null and zero”*).

**Step 1 – tests first.** These tests are written for you. Run them against the stub — they must fail (red).

In [12]:
class TestAverage(unittest.TestCase):

    def test_several_values(self):
        self.assertEqual(average([2, 4, 6]), 4)

    def test_single_value(self):
        self.assertEqual(average([7]), 7)

    def test_empty_list_raises(self):
        with self.assertRaises(ValueError):
            average([])

def average(values):        # stub: deliberately incomplete
    pass

run_tests(TestAverage);

test_empty_list_raises (__main__.TestAverage) ... 

FAIL


test_several_values (__main__.TestAverage) ... 

FAIL


test_single_value (__main__.TestAverage) ... 

FAIL


FAIL: test_empty_list_raises (__main__.TestAverage)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_16180\3318155443.py", line 10, in test_empty_list_raises
    with self.assertRaises(ValueError):
AssertionError: ValueError not raised



FAIL: test_several_values (__main__.TestAverage)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_16180\3318155443.py", line 4, in test_several_values
    self.assertEqual(average([2, 4, 6]), 4)
AssertionError: None != 4



FAIL: test_single_value (__main__.TestAverage)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_16180\3318155443.py", line 7, in test_single_value
    self.assertEqual(average([7]), 7)
AssertionError: None != 7



----------------------------------------------------------------------
Ran 3 tests in 0.003s

FAILED (failures=3)



==> 3 run, 3 failures, 0 errors


**✏️ TODO 5a (green).** Re-define `average` so that **all three** tests pass. Rerun the tests.

**✏️ TODO 5b (new mini-unit).** Add a test `test_negative_values` (e.g. `[-2, 2]` → `0`) *before* changing any code. Does it pass immediately? What does that tell you?

**✏️ TODO 5c (refactor).** If your code can be simpler, simplify it, then rerun all tests.

In [13]:
# TODO 5a (green): first working version, written step by step
def average(values):
    if len(values) == 0:
        raise ValueError("average() of an empty list is undefined")
    if len(values) == 1:            # "one is different"
        return values[0]
    total = 0
    for v in values:
        total += v
    return total / len(values)

run_tests(TestAverage);

test_empty_list_raises (__main__.TestAverage) ... 

ok


test_several_values (__main__.TestAverage) ... 

ok


test_single_value (__main__.TestAverage) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK



==> 3 run, 0 failures, 0 errors


**TODO 5b.** Tes baru `test_negative_values` ditambahkan **sebelum** kode `average` diubah:

In [14]:
class TestAverage(TestAverage):          # keeps the 3 earlier tests, adds a new one
    def test_negative_values(self):
        self.assertEqual(average([-2, 2]), 0)

run_tests(TestAverage);

test_empty_list_raises (__main__.TestAverage) ... 

ok


test_negative_values (__main__.TestAverage) ... 

ok


test_several_values (__main__.TestAverage) ... 

ok


test_single_value (__main__.TestAverage) ... 

ok


----------------------------------------------------------------------
Ran 4 tests in 0.004s

OK



==> 4 run, 0 failures, 0 errors


**Jawaban 5b.** Tes ini **langsung lulus** tanpa perubahan kode. Artinya implementasi yang ada sudah menangani partisi "nilai negatif", karena penjumlahan dan pembagian tidak memperlakukan angka negatif secara khusus. Tes ini tidak memaksa kita menulis kode baru, tetapi tetap berguna sebagai **regression test**. Pelajarannya: tes yang langsung hijau belum tentu bisa gagal. Kita perlu memastikan tes itu bisa merah, misalnya dengan merusak kode sementara (seperti mutation check di Exercise 2).

**TODO 5c (refactor).** Kasus khusus untuk satu nilai tidak perlu, karena `sum([7]) / 1 == 7`. Begitu juga loop manual, yang bisa diganti `sum()`:

In [15]:
def average(values):
    if not values:
        raise ValueError("average() of an empty list is undefined")
    return sum(values) / len(values)

run_tests(TestAverage);

test_empty_list_raises (__main__.TestAverage) ... 

ok


test_negative_values (__main__.TestAverage) ... 

ok


test_several_values (__main__.TestAverage) ... 

ok


test_single_value (__main__.TestAverage) ... 

ok


----------------------------------------------------------------------
Ran 4 tests in 0.004s

OK



==> 4 run, 0 failures, 0 errors


## Exercise 6 — Risk-based security testing

Risk 1 in Table 9.11 (*attacker gains access using authorized credentials*) can arise because *the user has set a weak password*. The lecture suggests a test that checks that the password-setting code **always checks password strength**.

We define our **own** policy for this exercise (it is an illustration, not a standard):

1. at least **12** characters;
2. not in the list `COMMON_PASSWORDS` (case-insensitive);
3. contains characters from at least **3** of these 4 classes: lowercase, uppercase, digit, symbol.

The tests from the lecture slide, plus a few more, are ready. **✏️ TODO 6.** Implement `is_strong_password` so they all pass.

In [16]:
COMMON_PASSWORDS = {"password", "123456", "qwerty", "letmein", "password123", "admin"}

def is_strong_password(pw):
    """Policy: >= 12 chars, not a common password (case-insensitive),
    and characters from at least 3 of: lowercase, uppercase, digit, symbol."""
    if not isinstance(pw, str):
        return False
    if len(pw) < 12:
        return False
    if pw.lower() in COMMON_PASSWORDS:
        return False
    classes = [
        any(c.islower() for c in pw),
        any(c.isupper() for c in pw),
        any(c.isdigit() for c in pw),
        any(not c.isalnum() for c in pw),   # symbol (anything not letter/digit)
    ]
    return sum(classes) >= 3

class TestPasswordPolicy(unittest.TestCase):
    # from the lecture slide
    def test_common_password_rejected(self):
        self.assertFalse(is_strong_password('password'))

    def test_short_password_rejected(self):
        self.assertFalse(is_strong_password('Ab1!'))

    def test_long_mixed_password_accepted(self):
        self.assertTrue(is_strong_password('Tr0ub4dor&3-horse'))

    # extra tests: edges and partitions of our policy
    def test_exactly_12_chars_three_classes_accepted(self):
        self.assertTrue(is_strong_password('abcdefgh12AB'))

    def test_11_chars_rejected(self):
        self.assertFalse(is_strong_password('abcdefgh12A'))

    def test_long_but_one_class_rejected(self):
        self.assertFalse(is_strong_password('abcdefghijklmnop'))

    def test_common_password_case_insensitive(self):
        self.assertFalse(is_strong_password('PASSWORD123'))

    def test_empty_rejected(self):
        self.assertFalse(is_strong_password(''))

run_tests(TestPasswordPolicy);

test_11_chars_rejected (__main__.TestPasswordPolicy) ... 

ok


test_common_password_case_insensitive (__main__.TestPasswordPolicy) ... 

ok


test_common_password_rejected (__main__.TestPasswordPolicy) ... 

ok


... (8 outputs left out)

test_short_password_rejected (__main__.TestPasswordPolicy) ... 

ok


----------------------------------------------------------------------
Ran 8 tests in 0.006s

OK



==> 8 run, 0 failures, 0 errors


## Exercise 7 — Code review with a checklist

You are the reviewer. Use the Python checklist from Table 9.13 on the function below:

* Are meaningful variable and function names used?
* Have all data errors been considered and tests written for them?
* Are all exceptions explicitly handled?
* Are default function parameters used?
* Are types used consistently?
* Is the indentation level correct?

**✏️ TODO 7a.** Temuan review:

1. **Nama tidak bermakna.** `f`, `x`, dan `l` tidak menjelaskan apa-apa. Huruf `l` juga mudah tertukar dengan angka `1`. Sebaiknya `add_score`, `score`, `scores`.
2. **Default parameter berbahaya (mutable default).** `l=[]` hanya dibuat **satu kali** saat fungsi didefinisikan, sehingga semua pemanggilan tanpa argumen kedua memakai list yang sama dan skor "bocor" antar-pemanggilan. Seharusnya `scores=None`, lalu buat list baru di dalam fungsi.
3. **Tipe tidak konsisten.** List bisa berisi angka *dan* string `"invalid"`. Kode yang nanti menghitung rata-rata dari list itu akan error atau salah hitung.
4. **Data error tidak ditangani secara eksplisit.** Skor di luar 0–100 tidak ditolak, hanya diam-diam dicatat sebagai `"invalid"`. Input bukan angka (misalnya `"80"`) memicu `TypeError` yang tidak disengaja dari perbandingan `str >= int`, tanpa pesan yang jelas. `True` bahkan diterima sebagai skor 1. Fungsi seharusnya melempar `ValueError`/`TypeError` dengan pesan yang jelas.
5. **Tidak ada tes** untuk kasus-kasus data error tersebut.
6. Kondisi `x >= 0 and x <= 100` bisa ditulis lebih jelas sebagai `0 <= x <= 100`. Indentasinya sendiri sudah benar.

In [17]:
def f(x, l=[]):
    # adds a student score (0-100) to a list and returns the list
    if x >= 0 and x <= 100:
        l.append(x)
    else:
        l.append("invalid")
    return l

**✏️ TODO 7b.** These tests describe the behaviour the reviewer and programmer agree on. Run them against `f` to see which issues they reveal, then write a fixed `add_score` that passes them all.

In [18]:
def add_score(score, scores=None):
    return f(score, scores) if scores is not None else f(score)   # ✏️ TODO: replace with a fixed version

class TestAddScore(unittest.TestCase):

    def test_calls_do_not_share_state(self):
        first = add_score(50)
        second = add_score(60)
        self.assertEqual(second, [60])

    def test_appends_to_given_list(self):
        self.assertEqual(add_score(70, [10]), [10, 70])

    def test_out_of_range_raises_value_error(self):
        with self.assertRaises(ValueError):
            add_score(101, [])

    def test_non_number_raises_type_error(self):
        with self.assertRaises(TypeError):
            add_score("80", [])

    def test_boundaries_accepted(self):
        self.assertEqual(add_score(0, []), [0])
        self.assertEqual(add_score(100, []), [100])

run_tests(TestAddScore);

test_appends_to_given_list (__main__.TestAddScore) ... 

ok


test_boundaries_accepted (__main__.TestAddScore) ... 

ok


test_calls_do_not_share_state (__main__.TestAddScore) ... 

FAIL


... (4 outputs left out)

FAIL: test_calls_do_not_share_state (__main__.TestAddScore)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_16180\201730034.py", line 9, in test_calls_do_not_share_state
    self.assertEqual(second, [60])
AssertionError: Lists differ: [50, 60] != [60]

First differing element 0:
50
60

First list contains 1 additional elements.
First extra element 1:
60

- [50, 60]
+ [60]



FAIL: test_out_of_range_raises_value_error (__main__.TestAddScore)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_16180\201730034.py", line 15, in test_out_of_range_raises_value_error
    with self.assertRaises(ValueError):
AssertionError: ValueError not raised



----------------------------------------------------------------------
Ran 5 tests in 0.006s

FAILED (failures=2)



==> 5 run, 2 failures, 0 errors


Hasil di atas memakai `f` yang asli. Dua tes gagal: `test_calls_do_not_share_state` (masalah *mutable default*, state bocor antar-pemanggilan) dan `test_out_of_range_raises_value_error` (skor 101 hanya dicatat sebagai `"invalid"`). `test_non_number_raises_type_error` lulus, tetapi hanya **kebetulan**: `TypeError`-nya datang dari perbandingan `"80" >= 0`, bukan dari pemeriksaan yang disengaja. `f(True)` pun masih diterima. Berikut versi yang sudah diperbaiki:

In [19]:
def add_score(score, scores=None):
    """Append a student score (0-100) to `scores` and return the list.

    Raises TypeError if score is not an int/float (bool is rejected too),
    and ValueError if it is outside 0..100.
    """
    if isinstance(score, bool) or not isinstance(score, (int, float)):
        raise TypeError(f"score must be a number, got {type(score).__name__}")
    if not 0 <= score <= 100:
        raise ValueError(f"score must be between 0 and 100, got {score}")
    if scores is None:          # new list for every call -> no shared state
        scores = []
    scores.append(score)
    return scores

run_tests(TestAddScore);

test_appends_to_given_list (__main__.TestAddScore) ... 

ok


test_boundaries_accepted (__main__.TestAddScore) ... 

ok


test_calls_do_not_share_state (__main__.TestAddScore) ... 

ok


... (2 outputs left out)

test_out_of_range_raises_value_error (__main__.TestAddScore) ... 

ok


----------------------------------------------------------------------
Ran 5 tests in 0.004s

OK



==> 5 run, 0 failures, 0 errors


## Exercise 8 — Running tests from files (Program 9.4)

In a real project tests live in files such as `test_namecheck.py`, and a runner *discovers* them. The next cells write a small project into a folder and run Program 9.4 on it.

In [20]:
import os, pathlib, subprocess, sys, textwrap

proj = pathlib.Path("testing_demo")
proj.mkdir(exist_ok=True)

(proj / "RE_checker.py").write_text(textwrap.dedent("""
    import re

    def namecheck(s):
        namex = r"^[a-zA-Z][a-zA-Z-']{1,39}$"
        if re.match(namex, s):
            if re.search("'.*'", s) or re.search("--", s):
                return False
            else:
                return True
        else:
            return False
"""))

(proj / "test_namecheck.py").write_text(textwrap.dedent("""
    import unittest
    from RE_checker import namecheck

    class TestNameCheck(unittest.TestCase):
        def test_alphaname(self):
            self.assertTrue(namecheck('Sommerville'))
        def test_namewithquote(self):
            self.assertTrue(namecheck("O'Reilly"))
        def test_nametooshort(self):
            self.assertFalse(namecheck('S'))
"""))

# Program 9.4, saved as a script
(proj / "run_tests.py").write_text(textwrap.dedent("""
    import unittest

    loader = unittest.TestLoader()
    # Find the test files in the current directory
    tests = loader.discover('.')
    # Specify the level of information provided by the test runner
    testRunner = unittest.runner.TextTestRunner(verbosity=2)
    testRunner.run(tests)
"""))

print(sorted(p.name for p in proj.iterdir()))

['RE_checker.py', '__pycache__', 'run_tests.py', 'test_edges.py', 'test_namecheck.py']


In [21]:
out = subprocess.run([sys.executable, "run_tests.py"], cwd=proj, capture_output=True, text=True)
print(out.stderr)   # unittest writes its report to stderr

test_empty_string_rejected (test_edges.TestNameCheckEdges) ... ok
test_length_1_rejected (test_edges.TestNameCheckEdges) ... ok
test_length_2_accepted (test_edges.TestNameCheckEdges) ... ok
test_length_40_accepted (test_edges.TestNameCheckEdges) ... ok
test_length_41_rejected (test_edges.TestNameCheckEdges) ... ok
test_alphaname (test_namecheck.TestNameCheck) ... ok
test_nametooshort (test_namecheck.TestNameCheck) ... ok
test_namewithquote (test_namecheck.TestNameCheck) ... ok

----------------------------------------------------------------------
Ran 8 tests in 0.000s

OK



**✏️ TODO 8.** Add a second file `test_edges.py` to `testing_demo/` containing your edge-case tests from Exercise 3 (re-run the cell above afterwards). Discovery picks up every file whose name matches `test*.py` — confirm that the number of tests run goes up.

Command-line equivalent: `python -m unittest` (run inside the folder).

In [22]:
(proj / "test_edges.py").write_text(textwrap.dedent("""
    import unittest
    from RE_checker import namecheck

    class TestNameCheckEdges(unittest.TestCase):
        def test_length_1_rejected(self):
            self.assertFalse(namecheck('A' * 1))
        def test_length_2_accepted(self):
            self.assertTrue(namecheck('A' * 2))
        def test_length_40_accepted(self):
            self.assertTrue(namecheck('A' * 40))
        def test_length_41_rejected(self):
            self.assertFalse(namecheck('A' * 41))
        def test_empty_string_rejected(self):
            self.assertFalse(namecheck(''))
"""))

out = subprocess.run([sys.executable, "run_tests.py"], cwd=proj, capture_output=True, text=True)
print(out.stderr)   # should now report 3 + 5 = 8 tests

test_empty_string_rejected (test_edges.TestNameCheckEdges) ... ok
test_length_1_rejected (test_edges.TestNameCheckEdges) ... ok
test_length_2_accepted (test_edges.TestNameCheckEdges) ... ok
test_length_40_accepted (test_edges.TestNameCheckEdges) ... ok
test_length_41_rejected (test_edges.TestNameCheckEdges) ... ok
test_alphaname (test_namecheck.TestNameCheck) ... ok
test_nametooshort (test_namecheck.TestNameCheck) ... ok
test_namewithquote (test_namecheck.TestNameCheck) ... ok

----------------------------------------------------------------------
Ran 8 tests in 0.001s

OK



---
# Bagian B — Mini-proyek: fungsi `nilai_huruf`

Anda akan menerapkan partisi ekuivalen, edge case, pola Arrange–Action–Assert, dan TDD pada satu fungsi kecil.

### Spesifikasi (khusus untuk tugas ini, bukan aturan nilai resmi)

`nilai_huruf(skor)` menerima skor angka dan mengembalikan nilai huruf:

| Skor | Hasil |
|---|---|
| 80 ≤ skor ≤ 100 | `'A'` |
| 70 ≤ skor < 80 | `'B'` |
| 60 ≤ skor < 70 | `'C'` |
| 50 ≤ skor < 60 | `'D'` |
| 0 ≤ skor < 50 | `'E'` |
| skor < 0 atau skor > 100 | `ValueError` |
| bukan `int`/`float` (termasuk `bool`, `str`, `None`) | `TypeError` |

### B1. Partisi ekuivalen dan nilai batas

| Partisi | Valid / tidak valid | Contoh input | Hasil yang diharapkan |
|---|---|---|---|
| P1: 80 ≤ skor ≤ 100 | Valid | `90` | `'A'` |
| P2: 70 ≤ skor < 80 | Valid | `75` | `'B'` |
| P3: 60 ≤ skor < 70 | Valid | `65` | `'C'` |
| P4: 50 ≤ skor < 60 | Valid | `55` | `'D'` |
| P5: 0 ≤ skor < 50 | Valid | `25` | `'E'` |
| P6: skor bertipe `float` di dalam 0–100 | Valid | `79.5` | `'B'` |
| P7: skor < 0 | Tidak valid | `-50` | `ValueError` |
| P8: skor > 100 | Tidak valid | `150` | `ValueError` |
| P9: `bool` (subclass dari `int`) | Tidak valid | `True` | `TypeError` |
| P10: `str` | Tidak valid | `"80"` | `TypeError` |
| P11: `None` | Tidak valid | `None` | `TypeError` |
| P12: tipe lain (mis. `list`) | Tidak valid | `[80]` | `TypeError` |

**Nilai batas yang diuji** (di setiap batas diambil nilai *pada* batas dan nilai *tepat di sebelahnya*):

1. `-1` → `ValueError`
2. `-0.01` → `ValueError`
3. `0` → `'E'`
4. `49.99` → `'E'`
5. `50` → `'D'`
6. `59.99` → `'D'`
7. `60` → `'C'`
8. `69.99` → `'C'`
9. `70` → `'B'`
10. `79.99` → `'B'`
11. `80` → `'A'`
12. `100` → `'A'`
13. `100.01` → `ValueError`
14. `101` → `ValueError`

### B2. Kembangkan `nilai_huruf` dengan TDD

Lakukan **minimal 3 siklus TDD**. Untuk setiap siklus, buat sel-sel berikut secara berurutan dan **biarkan output-nya terlihat**:

1. Sel markdown: `Siklus n – <apa yang ditambahkan>`.
2. Sel kode: tes baru (tambahkan ke kelas tes) dan jalankan → hasil **merah** (gagal).
3. Sel kode: kode baru/perbaikan `nilai_huruf` dan jalankan ulang **semua** tes → hasil **hijau**.

Contoh pembagian siklus: (1) skor valid → huruf; (2) skor di luar 0–100 → `ValueError`; (3) tipe salah → `TypeError`. Siklus pertama sudah dimulai di bawah sebagai contoh format.

#### Siklus 1 – skor valid menghasilkan huruf yang benar

In [23]:
class TestNilaiHuruf(unittest.TestCase):

    def test_skor_90_mendapat_A(self):
        # Arrange
        skor = 90
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'A')

def nilai_huruf(skor):      # stub: sengaja belum lengkap
    pass

run_tests(TestNilaiHuruf);

test_skor_90_mendapat_A (__main__.TestNilaiHuruf) ... 

FAIL


FAIL: test_skor_90_mendapat_A (__main__.TestNilaiHuruf)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_16180\3346820054.py", line 9, in test_skor_90_mendapat_A
    self.assertEqual(hasil, 'A')
AssertionError: None != 'A'



----------------------------------------------------------------------
Ran 1 test in 0.001s

FAILED (failures=1)



==> 1 run, 1 failures, 0 errors


In [24]:
# Siklus 1 (hijau): implementasi paling sederhana yang membuat tes 90 -> 'A' lulus.
def nilai_huruf(skor):
    if skor >= 80:
        return 'A'

run_tests(TestNilaiHuruf);

test_skor_90_mendapat_A (__main__.TestNilaiHuruf) ... 

ok


----------------------------------------------------------------------
Ran 1 test in 0.001s

OK



==> 1 run, 0 failures, 0 errors


#### Siklus 2 – huruf B sampai E dan semua nilai batas di dalam 0–100

In [25]:
class TestNilaiHuruf(TestNilaiHuruf):     # mewarisi tes siklus sebelumnya, lalu menambah tes baru

    # --- satu wakil per partisi valid ---
    def test_skor_75_mendapat_B(self):
        # Arrange
        skor = 75
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'B')

    def test_skor_65_mendapat_C(self):
        # Arrange
        skor = 65
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'C')

    def test_skor_55_mendapat_D(self):
        # Arrange
        skor = 55
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'D')

    def test_skor_25_mendapat_E(self):
        # Arrange
        skor = 25
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'E')

    def test_skor_float_79_5_mendapat_B(self):
        # Arrange
        skor = 79.5
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'B')

    # --- nilai batas ---
    def test_batas_0_mendapat_E(self):
        # Arrange
        skor = 0
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'E')

    def test_batas_49_99_mendapat_E(self):
        # Arrange
        skor = 49.99
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'E')

    def test_batas_50_mendapat_D(self):
        # Arrange
        skor = 50
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'D')

    def test_batas_59_99_mendapat_D(self):
        # Arrange
        skor = 59.99
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'D')

    def test_batas_60_mendapat_C(self):
        # Arrange
        skor = 60
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'C')

    def test_batas_69_99_mendapat_C(self):
        # Arrange
        skor = 69.99
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'C')

    def test_batas_70_mendapat_B(self):
        # Arrange
        skor = 70
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'B')

    def test_batas_79_99_mendapat_B(self):
        # Arrange
        skor = 79.99
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'B')

    def test_batas_80_mendapat_A(self):
        # Arrange
        skor = 80
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'A')

    def test_batas_100_mendapat_A(self):
        # Arrange
        skor = 100
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'A')

run_tests(TestNilaiHuruf);   # MERAH: skor < 80 masih menghasilkan None

test_batas_0_mendapat_E (__main__.TestNilaiHuruf) ... 

FAIL


test_batas_100_mendapat_A (__main__.TestNilaiHuruf) ... 

ok


test_batas_49_99_mendapat_E (__main__.TestNilaiHuruf) ... 

FAIL


... (37 outputs left out)

FAIL: test_skor_75_mendapat_B (__main__.TestNilaiHuruf)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_16180\811570563.py", line 10, in test_skor_75_mendapat_B
    self.assertEqual(hasil, 'B')
AssertionError: None != 'B'



FAIL: test_skor_float_79_5_mendapat_B (__main__.TestNilaiHuruf)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_16180\811570563.py", line 42, in test_skor_float_79_5_mendapat_B
    self.assertEqual(hasil, 'B')
AssertionError: None != 'B'



----------------------------------------------------------------------
Ran 16 tests in 0.011s

FAILED (failures=13)



==> 16 run, 13 failures, 0 errors


In [26]:
def nilai_huruf(skor):
    if skor >= 80:
        return 'A'
    elif skor >= 70:
        return 'B'
    elif skor >= 60:
        return 'C'
    elif skor >= 50:
        return 'D'
    else:
        return 'E'

run_tests(TestNilaiHuruf);   # HIJAU

test_batas_0_mendapat_E (__main__.TestNilaiHuruf) ... 

ok


test_batas_100_mendapat_A (__main__.TestNilaiHuruf) ... 

ok


test_batas_49_99_mendapat_E (__main__.TestNilaiHuruf) ... 

ok


... (24 outputs left out)

test_skor_float_79_5_mendapat_B (__main__.TestNilaiHuruf) ... 

ok


----------------------------------------------------------------------
Ran 16 tests in 0.015s

OK



==> 16 run, 0 failures, 0 errors


#### Siklus 3 – skor di luar 0–100 menghasilkan `ValueError`

In [27]:
class TestNilaiHuruf(TestNilaiHuruf):

    def test_partisi_negatif_minus_50_ValueError(self):
        # Arrange
        skor = -50
        # Action + Assert
        with self.assertRaises(ValueError):
            nilai_huruf(skor)

    def test_batas_minus_1_ValueError(self):
        # Arrange
        skor = -1
        # Action + Assert
        with self.assertRaises(ValueError):
            nilai_huruf(skor)

    def test_batas_minus_0_01_ValueError(self):
        # Arrange
        skor = -0.01
        # Action + Assert
        with self.assertRaises(ValueError):
            nilai_huruf(skor)

    def test_batas_100_01_ValueError(self):
        # Arrange
        skor = 100.01
        # Action + Assert
        with self.assertRaises(ValueError):
            nilai_huruf(skor)

    def test_batas_101_ValueError(self):
        # Arrange
        skor = 101
        # Action + Assert
        with self.assertRaises(ValueError):
            nilai_huruf(skor)

    def test_partisi_di_atas_150_ValueError(self):
        # Arrange
        skor = 150
        # Action + Assert
        with self.assertRaises(ValueError):
            nilai_huruf(skor)

run_tests(TestNilaiHuruf);   # MERAH: -1 -> 'E', 101 -> 'A', tidak ada ValueError

test_batas_0_mendapat_E (__main__.TestNilaiHuruf) ... 

ok


test_batas_100_01_ValueError (__main__.TestNilaiHuruf) ... 

FAIL


test_batas_100_mendapat_A (__main__.TestNilaiHuruf) ... 

ok


... (42 outputs left out)

FAIL: test_partisi_di_atas_150_ValueError (__main__.TestNilaiHuruf)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_16180\3190289514.py", line 42, in test_partisi_di_atas_150_ValueError
    with self.assertRaises(ValueError):
AssertionError: ValueError not raised



FAIL: test_partisi_negatif_minus_50_ValueError (__main__.TestNilaiHuruf)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_16180\3190289514.py", line 7, in test_partisi_negatif_minus_50_ValueError
    with self.assertRaises(ValueError):
AssertionError: ValueError not raised



----------------------------------------------------------------------
Ran 22 tests in 0.019s

FAILED (failures=6)



==> 22 run, 6 failures, 0 errors


In [28]:
def nilai_huruf(skor):
    if skor < 0 or skor > 100:
        raise ValueError("skor harus di antara 0 dan 100")
    if skor >= 80:
        return 'A'
    elif skor >= 70:
        return 'B'
    elif skor >= 60:
        return 'C'
    elif skor >= 50:
        return 'D'
    else:
        return 'E'

run_tests(TestNilaiHuruf);   # HIJAU

test_batas_0_mendapat_E (__main__.TestNilaiHuruf) ... 

ok


test_batas_100_01_ValueError (__main__.TestNilaiHuruf) ... 

ok


test_batas_100_mendapat_A (__main__.TestNilaiHuruf) ... 

ok


... (36 outputs left out)

test_skor_float_79_5_mendapat_B (__main__.TestNilaiHuruf) ... 

ok


----------------------------------------------------------------------
Ran 22 tests in 0.016s

OK



==> 22 run, 0 failures, 0 errors


#### Siklus 4 – tipe yang salah menghasilkan `TypeError`

In [29]:
class TestNilaiHuruf(TestNilaiHuruf):

    def test_bool_True_TypeError(self):
        # Arrange
        skor = True
        # Action + Assert
        with self.assertRaises(TypeError):
            nilai_huruf(skor)

    def test_string_TypeError(self):
        # Arrange
        skor = "80"
        # Action + Assert
        with self.assertRaises(TypeError):
            nilai_huruf(skor)

    def test_None_TypeError(self):
        # Arrange
        skor = None
        # Action + Assert
        with self.assertRaises(TypeError):
            nilai_huruf(skor)

    def test_list_TypeError(self):
        # Arrange
        skor = [80]
        # Action + Assert
        with self.assertRaises(TypeError):
            nilai_huruf(skor)

run_tests(TestNilaiHuruf);   # MERAH: True dianggap angka 1 -> 'E'

test_None_TypeError (__main__.TestNilaiHuruf) ... 

ok


test_batas_0_mendapat_E (__main__.TestNilaiHuruf) ... 

ok


test_batas_100_01_ValueError (__main__.TestNilaiHuruf) ... 

ok


... (45 outputs left out)

ok


FAIL: test_bool_True_TypeError (__main__.TestNilaiHuruf)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_16180\3635938962.py", line 7, in test_bool_True_TypeError
    with self.assertRaises(TypeError):
AssertionError: TypeError not raised



----------------------------------------------------------------------
Ran 26 tests in 0.019s

FAILED (failures=1)



==> 26 run, 1 failures, 0 errors


Catatan: tes `"80"`, `None`, dan `[80]` sudah **lulus sebelum** kode diubah, karena Python sendiri melempar `TypeError` saat membandingkan `str`/`None`/`list` dengan `0`. Tes-tes itu lulus "kebetulan", bukan karena kode kita sengaja memeriksa tipe. Hanya `True` yang benar-benar merah, karena `bool` adalah subclass dari `int`. Pemeriksaan tipe eksplisit di bawah membuat perilaku ini disengaja dan pesannya jelas.

In [30]:
def nilai_huruf(skor):
    if isinstance(skor, bool) or not isinstance(skor, (int, float)):
        raise TypeError("skor harus berupa angka (int/float)")
    if skor < 0 or skor > 100:
        raise ValueError("skor harus di antara 0 dan 100")
    if skor >= 80:
        return 'A'
    elif skor >= 70:
        return 'B'
    elif skor >= 60:
        return 'C'
    elif skor >= 50:
        return 'D'
    else:
        return 'E'

run_tests(TestNilaiHuruf);   # HIJAU

test_None_TypeError (__main__.TestNilaiHuruf) ... 

ok


test_batas_0_mendapat_E (__main__.TestNilaiHuruf) ... 

ok


test_batas_100_01_ValueError (__main__.TestNilaiHuruf) ... 

ok


... (44 outputs left out)

test_string_TypeError (__main__.TestNilaiHuruf) ... 

ok


----------------------------------------------------------------------
Ran 26 tests in 0.020s

OK



==> 26 run, 0 failures, 0 errors


### B3. Kumpulan tes lengkap

✏️ Pada akhir B2, kelas `TestNilaiHuruf` Anda harus berisi **minimal 15 tes**, masing-masing dengan komentar `# Arrange`, `# Action`, `# Assert`. Tes harus mencakup semua partisi dan semua nilai batas dari B1. Jalankan sel di bawah untuk memastikan semuanya lulus terhadap implementasi Anda.

In [31]:
jumlah = unittest.defaultTestLoader.loadTestsFromTestCase(TestNilaiHuruf).countTestCases()
print("Jumlah tes di TestNilaiHuruf:", jumlah)
run_tests(TestNilaiHuruf);

test_None_TypeError (__main__.TestNilaiHuruf) ... 

ok


test_batas_0_mendapat_E (__main__.TestNilaiHuruf) ... 

ok


test_batas_100_01_ValueError (__main__.TestNilaiHuruf) ... 

ok


... (45 outputs left out)

test_string_TypeError (__main__.TestNilaiHuruf) ... 

ok


----------------------------------------------------------------------
Ran 26 tests in 0.020s

OK



==> 26 run, 0 failures, 0 errors


### B4. Tes Anda melawan implementasi orang lain

Seorang rekan menulis `nilai_huruf_v2` di bawah ini. Implementasi ini **mengandung bug**. Jangan ubah kodenya.

✏️ Jalankan tes Anda terhadap `nilai_huruf_v2` memakai `uji_dengan`. Lalu, di sel markdown setelahnya, jelaskan **setiap** tes yang gagal: bug apa yang ditemukan, di baris mana, dan nilai batas atau partisi mana yang menemukannya. Jika tidak ada tes yang gagal, tambahkan tes sampai bug-nya tertangkap.

In [32]:
def nilai_huruf_v2(skor):
    if isinstance(skor, bool) or not isinstance(skor, (int, float)):
        raise TypeError("skor harus berupa angka")
    if skor < 0 or skor > 101:
        raise ValueError("skor harus di antara 0 dan 100")
    if skor > 80:
        return 'A'
    elif skor >= 70:
        return 'B'
    elif skor >= 60:
        return 'C'
    elif skor >= 50:
        return 'D'
    else:
        return 'E'

def uji_dengan(fungsi, *kelas_tes):
    """Jalankan kelas tes dengan `nilai_huruf` sementara diganti `fungsi`."""
    g = globals()
    asli = g.get('nilai_huruf')
    g['nilai_huruf'] = fungsi
    try:
        return run_tests(*kelas_tes, verbosity=1)
    finally:
        g['nilai_huruf'] = asli

uji_dengan(nilai_huruf_v2, TestNilaiHuruf);

.

.

F

.

F

.

... (21 outputs left out)

FAIL: test_batas_101_ValueError (__main__.TestNilaiHuruf)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_16180\3190289514.py", line 35, in test_batas_101_ValueError
    with self.assertRaises(ValueError):
AssertionError: ValueError not raised



FAIL: test_batas_80_mendapat_A (__main__.TestNilaiHuruf)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ASUS\AppData\Local\Temp\ipykernel_16180\811570563.py", line 115, in test_batas_80_mendapat_A
    self.assertEqual(hasil, 'A')
AssertionError: 'B' != 'A'
- B
+ A




----------------------------------------------------------------------
Ran 26 tests in 0.011s

FAILED (failures=3)



==> 26 run, 3 failures, 0 errors


✏️ **Analisis bug**

| Tes yang gagal | Bug yang ditemukan | Baris di `nilai_huruf_v2` | Partisi / nilai batas |
|---|---|---|---|
| `test_batas_80_mendapat_A` | Batas bawah nilai A memakai `skor > 80`, padahal seharusnya `skor >= 80`. Akibatnya skor 80 tepat mendapat `'B'`, bukan `'A'`. | Baris 6: `if skor > 80:` | Nilai batas **80** (batas bawah P1) |
| `test_batas_100_01_ValueError` | Batas atas rentang valid memakai `skor > 101`, padahal seharusnya `skor > 100`. Skor 100.01 tidak ditolak dan malah mendapat `'A'`. | Baris 4: `if skor < 0 or skor > 101:` | Nilai batas **100.01** (tepat di atas batas P1/P8) |
| `test_batas_101_ValueError` | Bug yang sama dengan baris 4: skor 101 lolos (`101 > 101` bernilai `False`) dan mendapat `'A'`. | Baris 4 | Nilai batas **101** |

Kedua bug ini termasuk **off-by-one** di batas partisi. Tes perwakilan partisi seperti `90`, `75`, atau `150` tidak menemukannya. Yang menemukannya justru tes **tepat pada atau di sebelah batas**. Ini bukti bahwa panduan *"test edge cases"* di Table 9.4 memang penting.

---
# Bagian C — Refleksi (sekitar 300 kata)

1. Dari pengalaman Bagian B, apa satu kelebihan dan satu kesulitan TDD yang Anda rasakan sendiri? Bandingkan dengan Table 9.10.
2. Bagaimana Anda memastikan tes yang Anda tulis **bisa gagal** (tidak seperti `test_shortname`)?

*Jawaban Anda:*

Kelebihan TDD yang paling terasa bagi saya adalah tes ditulis lebih dulu, sehingga saya terpaksa memikirkan spesifikasi secara rinci sebelum menulis kode. Saat menyusun tes untuk `nilai_huruf`, saya harus memutuskan apa yang terjadi tepat di skor 80, di 100.01, atau ketika inputnya `True`. Hal-hal seperti itu mudah terlewat kalau langsung menulis `if-elif`. Hasilnya terlihat di B4: tes batas yang saya tulis sejak awal langsung menangkap dua bug off-by-one di `nilai_huruf_v2` (`> 80` dan `> 101`). Ini sejalan dengan kelebihan TDD di Table 9.10, yaitu *systematic testing*, *code coverage* yang lebih baik, dan *regression testing*: setiap kali saya menambah pemeriksaan `ValueError` atau `TypeError`, semua tes lama ikut dijalankan ulang dan saya yakin tidak ada yang rusak. Kesulitannya, TDD terasa lambat dan berulang untuk fungsi sederhana. Saya menulis lebih dari 25 tes dengan pola Arrange–Action–Assert yang hampir sama, dan di siklus 1 saya sengaja menulis kode "terlalu sederhana" hanya agar satu tes lulus. Table 9.10 juga menyebut bahwa TDD sulit diterapkan kalau spesifikasinya belum jelas dan bisa membuat programmer terlalu fokus pada tes dibanding desain keseluruhan. Saya juga melihat bahwa tes yang lulus belum tentu benar: tes `"80"` dan `None` di siklus 4 sudah hijau sebelum kodenya ditulis.

Untuk memastikan tes bisa gagal, saya selalu menjalankannya dulu dalam keadaan **merah**, yaitu terhadap stub atau kode lama sebelum implementasi ditulis. Kalau sebuah tes langsung hijau, saya curigai, lalu cek apakah tes itu benar-benar memanggil fungsi yang diuji (kesalahan pada `test_shortname` adalah `assertTrue('Sx')` tanpa memanggil `namecheck`). Cara kedua adalah *mutation check* seperti di Exercise 2: sengaja merusak kode, misalnya mengganti `>=` dengan `>`, lalu memastikan ada tes yang gagal. B4 sebenarnya juga uji seperti ini, karena `nilai_huruf_v2` adalah "mutan" dari implementasi saya. Terakhir, setiap tes harus berisi assertion yang membandingkan hasil dengan nilai yang spesifik, bukan hanya `pass` atau sekadar memeriksa "truthy".